# Scenario 5: Claude Code for Continuous Integration

**Domains tested:** Claude Code Config (D3 -- 20%), Prompt Engineering (D4 -- 20%)

---

## Scenario Context

You are the CI/CD lead at **ShieldSec**, a cybersecurity company. Your team is integrating Claude Code into the CI pipeline for automated code reviews, test generation, and PR feedback. The system must produce actionable, low-noise feedback that developers trust.

**Production requirements:**
- Claude Code runs in non-interactive mode (`-p` flag) in CI jobs
- Output must be machine-parseable (`--output-format json`)
- Reviews must have explicit, categorical criteria (no vague "looks good" feedback)
- Multi-pass review: per-file analysis followed by cross-file architectural review
- Overnight batch analysis for comprehensive audits using the Batch API
- False positive rate must stay below 10% to maintain developer trust

**CI pipeline architecture:**
```
PR Created -> Per-file Review (Claude Code -p) -> Cross-file Review -> Post Results
                                                                           |
Nightly -> Batch API Audit (comprehensive) -> Generate Report -> Notify Team
```

---

In [ ]:
import anthropic
import json
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

---

## Part 1: Exam-Format Questions

---

### Question 1 (D3)

You need to run Claude Code in a CI/CD pipeline where there is no terminal for interactive input. Which command correctly invokes Claude Code for non-interactive use?

- **A)** `claude --no-interactive "Review this PR"`
- **B)** `echo "Review this PR" | claude`
- **C)** `claude --batch "Review this PR"`
- **D)** `claude -p "Review the changes in this PR for security issues" --output-format json`

In [ ]:
q1 = {
    "correct": "D",
    "explanation": (
        "The -p flag puts Claude Code in non-interactive (prompt) mode, which is required for CI "
        "environments. Combined with --output-format json, this produces machine-parseable output "
        "that CI scripts can process. Option A uses a non-existent flag. Option B pipes input but "
        "doesn't enable non-interactive mode properly. Option C uses a non-existent flag. "
        "This is a direct Domain 3 knowledge test: -p is the CI/CD mode flag for Claude Code."
    )
}
print(f"Correct: {q1['correct']}")
print(f"\nExplanation: {q1['explanation']}")

### Question 2 (D4 + D3)

Your CI review bot uses the prompt: "Review this code and report any issues." The false positive rate is 40%. Developers are ignoring the reviews. What is the most effective change?

- **A)** Add "Be very conservative" to the prompt
- **B)** Replace the vague prompt with explicit categorical criteria: specific DO flag / DON'T flag rules for each review category (security, null safety, error handling), and separate the categories into independent review passes
- **C)** Run the review 3 times and only report issues found in all 3 runs
- **D)** Switch to a faster model to reduce CI pipeline time

In [ ]:
q2 = {
    "correct": "B",
    "explanation": (
        "This directly applies the Module 1 lesson: explicit criteria over vague instructions. "
        "Categorical DO/DON'T rules give the model concrete criteria for what to flag and what "
        "to skip. Separating into independent review passes means each pass has focused criteria, "
        "reducing cross-category confusion. Option A is the classic exam trap -- 'be conservative' "
        "is not a precision control. Option C (consensus voting) reduces recall along with false "
        "positives. Option D doesn't address prompt quality. This tests Domain 4 (explicit criteria) "
        "applied in the Domain 3 context (CI configuration)."
    )
}
print(f"Correct: {q2['correct']}")
print(f"\nExplanation: {q2['explanation']}")

### Question 3 (D3 + D4)

Your multi-pass review system runs a per-file security review, then a cross-file architectural review. The cross-file review consistently misses issues that span module boundaries (e.g., authentication bypass where the middleware is defined in one file but misconfigured in another).

What is the most likely cause?

- **A)** The model cannot understand code spread across files
- **B)** The cross-file review is not receiving the per-file review findings as context, so it starts from scratch without knowing what was already identified
- **C)** The cross-file prompt says "review for architectural issues" instead of specifying exact cross-file patterns to check (e.g., "verify that every route has the auth middleware applied")
- **D)** Cross-file reviews are not possible with current models

In [ ]:
q3 = {
    "correct": "C",
    "explanation": (
        "The cross-file review needs explicit criteria for what cross-file patterns to check, "
        "just like per-file reviews need explicit criteria. 'Review for architectural issues' "
        "is vague -- the model needs specific checks like 'verify every route has auth middleware', "
        "'check that all database calls use the connection pool', 'ensure error types match across "
        "service boundaries.' Option B is also important (passing prior findings as context) but "
        "the ROOT cause is the vague cross-file prompt. Option A is wrong -- models can analyze "
        "multiple files. Option D is wrong. This tests D4 (explicit criteria) in the cross-file "
        "review context (D3)."
    )
}
print(f"Correct: {q3['correct']}")
print(f"\nExplanation: {q3['explanation']}")

### Question 4 (D3)

You want to run a comprehensive codebase audit overnight using Claude. The audit analyzes 500 files for security patterns. This is not time-sensitive but must be cost-effective. Which approach is best?

- **A)** Use the Anthropic Batch API to submit all 500 review requests at once, with results available within 24 hours at 50% cost reduction
- **B)** Run Claude Code with `-p` in a loop, one file at a time
- **C)** Use a single Claude Code session with all 500 files in context
- **D)** Run 500 parallel Claude Code processes

In [ ]:
q4 = {
    "correct": "A",
    "explanation": (
        "The Batch API is designed for non-time-sensitive workloads. It processes requests "
        "asynchronously with results available within 24 hours, at a 50% cost reduction compared "
        "to real-time API calls. This is ideal for overnight audits. Option B works but is slower "
        "and more expensive (full price per request). Option C would exceed context limits for 500 "
        "files. Option D would hit rate limits immediately. This tests Domain 3: knowing when to "
        "use the Batch API vs real-time processing."
    )
}
print(f"Correct: {q4['correct']}")
print(f"\nExplanation: {q4['explanation']}")

### Question 5 (D4)

Your per-file review prompt produces JSON output with this structure: `{"issues": [{"file": "...", "line": 42, "severity": "high", "issue": "..."}]}`. But 20% of the time, the model returns malformed JSON or includes markdown formatting around the JSON.

What is the most reliable fix?

- **A)** Use tool_use to force structured output -- define a tool whose input schema matches the desired output format, and the model will be constrained to produce valid JSON matching the schema
- **B)** Add "Return ONLY valid JSON" to the prompt
- **C)** Parse the output with a lenient JSON parser that strips markdown
- **D)** Lower the temperature to 0 to make output more deterministic

In [ ]:
q5 = {
    "correct": "A",
    "explanation": (
        "Tool use (function calling) forces the model to produce output that matches the tool's "
        "input schema. This is structurally guaranteed to be valid JSON matching your schema, "
        "unlike text-based JSON generation which can include markdown, extra text, or syntax "
        "errors. Option B helps but doesn't guarantee compliance. Option C is a workaround, not "
        "a fix. Option D reduces randomness but doesn't prevent formatting issues. This is a key "
        "Domain 4 concept: tool_use for structured output is more reliable than asking for JSON "
        "in a text prompt."
    )
}
print(f"Correct: {q5['correct']}")
print(f"\nExplanation: {q5['explanation']}")

### Question 6 (D3 + D4)

Your CI review system produces per-file results, then feeds them into a cross-file review. The cross-file review receives the original code AND the per-file findings. You notice the cross-file review sometimes contradicts per-file findings.

What is the best approach to using prior findings?

- **A)** Don't pass per-file findings to the cross-file review -- let it form independent conclusions
- **B)** Have the cross-file review validate per-file findings and overwrite contradictions
- **C)** Merge all findings into a single list without distinction
- **D)** Pass per-file findings as prior context with a clear instruction: "The following per-file issues were already identified. Your job is to find CROSS-FILE issues that individual file reviews cannot detect. Do not re-review issues already flagged."

In [ ]:
q6 = {
    "correct": "D",
    "explanation": (
        "Prior findings should be passed as context with explicit scope instructions. The cross-file "
        "review should know what was already found (to avoid duplication) and focus on issues that "
        "require multi-file analysis. Option A wastes the per-file work and may miss connections. "
        "Option B creates confusion about which review is authoritative. Option C loses the "
        "distinction between per-file and cross-file issues. This tests D3 (multi-pass CI design) "
        "and D4 (clear prompt scoping with prior context)."
    )
}
print(f"Correct: {q6['correct']}")
print(f"\nExplanation: {q6['explanation']}")

### Question 7 (D4 + D3)

Your security review prompt checks for SQL injection. It flags this code:

```python
query = f"SELECT * FROM users WHERE id = {user_id}"
```

But it also flags this code (false positive):

```python
query = "SELECT * FROM users WHERE id = %s"
cursor.execute(query, (user_id,))
```

The second example uses parameterized queries, which are safe. How do you fix this?

- **A)** Add "be careful about false positives" to the prompt
- **B)** Reduce the model temperature
- **C)** Add explicit DON'T flag criteria: "DO NOT flag parameterized queries where user input is passed as a separate parameter tuple/list to cursor.execute(), db.query(), or equivalent ORM methods. These are safe from SQL injection."
- **D)** Switch to a model fine-tuned for security analysis

In [ ]:
q7 = {
    "correct": "C",
    "explanation": (
        "Explicit DON'T flag criteria are the precision control for reducing false positives. "
        "By specifying that parameterized queries are safe, you teach the model the exact "
        "distinction between vulnerable string interpolation and safe parameterized queries. "
        "Option A is the classic vague instruction trap. Option B doesn't address the knowledge "
        "gap. Option D is expensive and unnecessary when the prompt can be improved. "
        "This directly applies Domain 4 (DO/DON'T criteria) to the CI review context (D3)."
    )
}
print(f"Correct: {q7['correct']}")
print(f"\nExplanation: {q7['explanation']}")

### Question 8 (D3 + D4)

You want Claude Code in CI to not only flag issues but also suggest fixes. However, auto-applying fixes in CI is risky. What is the safest approach?

- **A)** Let Claude Code auto-apply fixes and commit them to the PR
- **B)** Have Claude Code output suggested fixes in the review comments (as code suggestions) but do not auto-apply them -- developers review and apply manually
- **C)** Only flag issues, never suggest fixes
- **D)** Auto-apply fixes only for low-severity issues

In [ ]:
q8 = {
    "correct": "B",
    "explanation": (
        "Suggested fixes in review comments give developers actionable feedback while keeping "
        "humans in the loop. Auto-applying fixes (A, D) risks introducing bugs or unwanted "
        "changes. Option C provides no actionable guidance. The safest pattern: flag the issue, "
        "explain why it's a problem, and suggest a fix as a code comment that the developer "
        "reviews before applying. This tests the Domain 1 principle of human-in-the-loop "
        "for consequential actions, applied in the CI context (D3)."
    )
}
print(f"Correct: {q8['correct']}")
print(f"\nExplanation: {q8['explanation']}")

---

## Part 2: Hands-On Build

Build the multi-pass CI review system described in the scenario.

---

### Step 1: Define the per-file review with explicit criteria

In [ ]:
# Sample PR diff for review
PR_FILES = {
    "auth/middleware.py": {
        "diff": '''@@ -10,6 +10,15 @@
 def require_auth(func):
     def wrapper(request, *args, **kwargs):
         token = request.headers.get("Authorization")
+        if not token:
+            return JsonResponse({"error": "Unauthorized"}, status=401)
+        try:
+            payload = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])
+            request.user = payload
+        except jwt.ExpiredSignatureError:
+            return JsonResponse({"error": "Token expired"}, status=401)
+        except jwt.InvalidTokenError:
+            return JsonResponse({"error": "Invalid token"}, status=401)
         return func(request, *args, **kwargs)
     return wrapper''',
        "full_content": (
            'import jwt\n'
            'from django.http import JsonResponse\n'
            'SECRET_KEY = "hardcoded-secret-key-123"\n\n'
            'def require_auth(func):\n'
            '    def wrapper(request, *args, **kwargs):\n'
            '        token = request.headers.get("Authorization")\n'
            '        if not token:\n'
            '            return JsonResponse({"error": "Unauthorized"}, status=401)\n'
            '        try:\n'
            '            payload = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])\n'
            '            request.user = payload\n'
            '        except jwt.ExpiredSignatureError:\n'
            '            return JsonResponse({"error": "Token expired"}, status=401)\n'
            '        except jwt.InvalidTokenError:\n'
            '            return JsonResponse({"error": "Invalid token"}, status=401)\n'
            '        return func(request, *args, **kwargs)\n'
            '    return wrapper\n'
        )
    },
    "api/views.py": {
        "diff": '''@@ -1,4 +1,12 @@
+from auth.middleware import require_auth
 from django.http import JsonResponse
+from models import User
 
+@require_auth
 def get_user(request, user_id):
-    return JsonResponse({"user": "todo"})
+    query = f"SELECT * FROM users WHERE id = {user_id}"
+    user = db.execute(query).fetchone()
+    return JsonResponse({"user": dict(user)})''',
        "full_content": (
            'from auth.middleware import require_auth\n'
            'from django.http import JsonResponse\n'
            'from models import User\n\n'
            '@require_auth\n'
            'def get_user(request, user_id):\n'
            '    query = f"SELECT * FROM users WHERE id = {user_id}"\n'
            '    user = db.execute(query).fetchone()\n'
            '    return JsonResponse({"user": dict(user)})\n'
        )
    }
}

print(f"PR contains {len(PR_FILES)} changed files:")
for f in PR_FILES:
    print(f"  - {f}")

In [ ]:
REVIEW_TOOL = {
    "name": "report_review_findings",
    "description": "Report code review findings in structured format.",
    "input_schema": {
        "type": "object",
        "properties": {
            "findings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "file": {"type": "string"},
                        "line": {"type": "integer"},
                        "severity": {"type": "string", "enum": ["critical", "high", "medium", "low"]},
                        "category": {"type": "string", "enum": ["security", "null_safety", "error_handling", "style"]},
                        "issue": {"type": "string"},
                        "suggestion": {"type": "string"}
                    },
                    "required": ["file", "severity", "category", "issue", "suggestion"]
                }
            }
        },
        "required": ["findings"]
    }
}


def review_file(filename: str, file_data: dict) -> dict:
    """Per-file review with explicit criteria and structured output via tool_use."""
    prompt = f"""Review this code change for security issues.

FILE: {filename}

DIFF:
```
{file_data['diff']}
```

FULL FILE CONTENT:
```python
{file_data['full_content']}
```

REVIEW CRITERIA -- flag ONLY when:
1. SQL injection: string interpolation or concatenation used to build SQL queries with user input
   (DO NOT flag parameterized queries using %s, ?, or :param with separate parameter passing)
2. Hardcoded secrets: API keys, passwords, or secret keys hardcoded in source code
   (DO NOT flag environment variable lookups or config file references)
3. Missing authentication: public endpoints that access sensitive data without auth decorator
   (DO NOT flag internal utility functions or health check endpoints)
4. Insecure cryptography: use of deprecated algorithms (MD5, SHA1 for security), weak key sizes
   (DO NOT flag non-security uses of hashing like caching or checksums)

DO NOT flag:
- Style issues, naming conventions, or missing docstrings
- Performance concerns unless they create a security vulnerability
- Missing error handling for non-security-critical operations

Use the report_review_findings tool to submit your findings.
If no findings match the criteria, report an empty findings array."""

    response = client.messages.create(
        model=MODEL, max_tokens=2048,
        tools=[REVIEW_TOOL],
        messages=[{"role": "user", "content": prompt}]
    )

    # Extract tool use result
    for block in response.content:
        if block.type == "tool_use" and block.name == "report_review_findings":
            return block.input

    return {"findings": []}


# Run per-file reviews
all_per_file_findings = []
for filename, file_data in PR_FILES.items():
    print(f"\n=== Reviewing: {filename} ===")
    result = review_file(filename, file_data)
    findings = result.get("findings", [])
    all_per_file_findings.extend(findings)
    for f in findings:
        print(f"  [{f['severity'].upper()}] {f['category']}: {f['issue']}")
        print(f"    Suggestion: {f['suggestion']}")
    if not findings:
        print("  No issues found.")

print(f"\n=== Per-file review complete: {len(all_per_file_findings)} findings ===")

### Step 2: Cross-file architectural review

In [ ]:
def cross_file_review(files: dict, per_file_findings: list) -> dict:
    """Cross-file review that checks architectural patterns spanning multiple files."""

    all_code = "\n\n".join(
        f"=== {name} ===\n{data['full_content']}"
        for name, data in files.items()
    )

    prompt = f"""Perform a cross-file architectural review of this PR.

ALL FILES IN PR:
```python
{all_code}
```

PRIOR PER-FILE FINDINGS (already reported -- do not duplicate):
{json.dumps(per_file_findings, indent=2)}

CROSS-FILE REVIEW CRITERIA -- flag ONLY patterns that span multiple files:
1. Auth bypass: routes that access sensitive data but don't apply authentication middleware
2. Inconsistent error handling: different error response formats across related endpoints
3. Dependency issues: imports of modules that don't exist or circular dependencies
4. Secret leakage paths: secrets defined in one file used insecurely in another

DO NOT re-flag issues already in the per-file findings.
Focus exclusively on issues that require analyzing MULTIPLE files together.

Use the report_review_findings tool to submit your findings."""

    response = client.messages.create(
        model=MODEL, max_tokens=2048,
        tools=[REVIEW_TOOL],
        messages=[{"role": "user", "content": prompt}]
    )

    for block in response.content:
        if block.type == "tool_use" and block.name == "report_review_findings":
            return block.input

    return {"findings": []}


print("=== Cross-File Review ===")
cross_findings = cross_file_review(PR_FILES, all_per_file_findings)

for f in cross_findings.get("findings", []):
    print(f"  [{f['severity'].upper()}] {f['category']}: {f['issue']}")
    print(f"    Suggestion: {f['suggestion']}")

if not cross_findings.get("findings"):
    print("  No cross-file issues found.")

# Combined results
total_findings = all_per_file_findings + cross_findings.get("findings", [])
print(f"\n=== TOTAL: {len(total_findings)} findings ({len(all_per_file_findings)} per-file + {len(cross_findings.get('findings', []))} cross-file) ===")

### Step 3: Simulate Batch API for overnight audit

In [ ]:
print("=== Batch API Overnight Audit (Simulated) ===")
print()
print("In production, you would use the Anthropic Batch API:")
print()
print("# Create batch of review requests")
print("batch_requests = []")
print("for file_path in all_files_in_repo:")
print("    batch_requests.append({")
print('        "custom_id": file_path,')
print('        "params": {')
print(f'            "model": "{MODEL}",')
print('            "max_tokens": 2048,')
print('            "tools": [REVIEW_TOOL],')
print('            "messages": [{"role": "user", "content": review_prompt}]')
print("        }")
print("    })")
print()
print("# Submit batch")
print("batch = client.batches.create(requests=batch_requests)")
print(f"# batch.id -> 'batch_abc123'")
print(f"# Results available within 24 hours at 50% cost")
print()
print("# Poll for results")
print("result = client.batches.retrieve(batch.id)")
print("# result.status -> 'completed'")
print("# result.results -> list of review findings per file")
print()
print("Key benefits of Batch API for CI:")
print("  - 50% cost reduction for non-urgent workloads")
print("  - No rate limit concerns (batched processing)")
print("  - Ideal for overnight audits and comprehensive scans")
print("  - Results available within 24 hours")

---

## Part 3: Failure Injections

---

### Failure 1: Vague review prompt

In [ ]:
print("=== FAILURE INJECTION: Vague review prompt ===")
print()

VAGUE_PROMPT = f"""Review this code for issues. Be thorough but conservative.

```python
{PR_FILES['api/views.py']['full_content']}
```

Return findings as JSON array with file, severity, and issue."""

vague_findings = []
for run in range(3):
    response = client.messages.create(
        model=MODEL, max_tokens=1024,
        messages=[{"role": "user", "content": VAGUE_PROMPT}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        findings = json.loads(raw)
        vague_findings.append(len(findings))
        print(f"  Run {run+1}: {len(findings)} findings")
        for f in findings:
            print(f"    - {f.get('severity', '?')}: {f.get('issue', '?')[:80]}")
    except json.JSONDecodeError:
        vague_findings.append(-1)
        print(f"  Run {run+1}: MALFORMED JSON")

print(f"\nFindings per run: {vague_findings}")
print("\n--- DIAGNOSIS ---")
print("Vague prompts produce inconsistent results:")
print("  - Variable number of findings per run")
print("  - Mix of real issues and false positives")
print("  - Sometimes malformed JSON output")
print("  - 'Be conservative' does not control precision")
print("LESSON: Explicit criteria + tool_use = consistent, reliable CI reviews (D4).")

### Failure 2: No cross-file context

In [ ]:
print("=== FAILURE INJECTION: Cross-file review without prior findings ===")
print()

# Run cross-file review WITHOUT per-file findings as context
cross_no_context = cross_file_review(PR_FILES, [])  # empty prior findings

print("Cross-file review without prior context:")
for f in cross_no_context.get("findings", []):
    # Check if it duplicates per-file findings
    is_duplicate = any(
        pf.get("issue", "")[:30] in f.get("issue", "")
        for pf in all_per_file_findings
    )
    tag = " [DUPLICATE]" if is_duplicate else ""
    print(f"  [{f['severity'].upper()}] {f['issue'][:80]}{tag}")

print("\n--- DIAGNOSIS ---")
print("Without prior findings as context, the cross-file review:")
print("  - Re-flags per-file issues (duplicates)")
print("  - Dilutes attention away from true cross-file issues")
print("  - Produces noisy, redundant output")
print("LESSON: Pass prior findings as context with explicit scoping instructions (D3, D5).")

### Failure 3: Text-based JSON vs tool_use

In [ ]:
print("=== FAILURE INJECTION: Text JSON vs Tool Use ===")
print()

# Compare: text-based JSON output vs tool_use structured output
text_json_prompt = f"""Review this code for SQL injection:

```python
{PR_FILES['api/views.py']['full_content']}
```

Return ONLY a JSON object like: {{"findings": [{{"file": "...", "issue": "..."}}]}}"""

# Run 5 times and check for valid JSON
text_json_valid = 0
for i in range(5):
    response = client.messages.create(
        model=MODEL, max_tokens=512,
        messages=[{"role": "user", "content": text_json_prompt}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        parsed = json.loads(raw)
        text_json_valid += 1
    except json.JSONDecodeError:
        pass

# tool_use is always valid JSON by construction
tool_use_valid = 5  # tool_use output is guaranteed to match the schema

print(f"Text-based JSON: {text_json_valid}/5 valid responses")
print(f"Tool use output: {tool_use_valid}/5 valid responses (guaranteed by schema)")
print()
print("--- DIAGNOSIS ---")
print("Text-based JSON generation can produce:")
print("  - Markdown-wrapped JSON (```json ... ```)")
print("  - Extra explanatory text before/after the JSON")
print("  - Invalid JSON syntax")
print("Tool use guarantees valid JSON matching the schema.")
print("LESSON: Use tool_use for structured output in CI pipelines (D4).")

---

## Key Takeaways

| Concept | Domain | Lesson |
|---------|--------|--------|
| -p flag for CI | D3 | Use `-p` for non-interactive mode and `--output-format json` for machine-parseable output |
| Explicit review criteria | D4 | DO flag / DON'T flag lists produce consistent, low-noise reviews |
| Multi-pass review | D3, D4 | Per-file first, cross-file second, with prior findings as context |
| tool_use for structured output | D4 | Schema-enforced output is more reliable than text-based JSON |
| Batch API for audits | D3 | 50% cost reduction for non-urgent overnight analysis |
| Prior findings as context | D3, D5 | Pass earlier findings with explicit scoping to avoid duplication |
| Suggested fixes vs auto-apply | D1, D3 | Keep humans in the loop for consequential changes |